# 04. GeoAI Spatiotemporal Modeling & Recursive Projection for Land Subsidence

#### Notebook Overview & Research Methodology:
1. **Physical Environment & Dependencies**: Setup GPU environment, deterministic seeds, and geospatial libraries.
2. **Tensor Dataset Ingestion**: Load the 5D spatiotemporal tensor $(B, T=12, 64, 64, 3)$ without artificial spatial interpolation across coastal wetlands.
3. **Chronological Splitting with Transition Buffers**:
   - **Training Set**: January 2021 – December 2024
   - *Buffer 1*: January 2025 (prevents short-term target autocorrelation)
   - **Validation Set**: February 2025 – July 2025
   - *Buffer 2*: August 2025
   - **Testing Set**: September 2025 – April 2026
4. **Masked Loss Formulation**: Exclude decorrelated water bodies, aquaculture ponds, and intertidal flats from loss and backpropagation.
5. **Model Implementations**:
   - **ConvLSTM**: Simultaneous spatio-temporal recurrence ($3 \times 3$ convolutions, Layer 1: 64 filters, Layer 2: 32 filters, pure `tanh` to ensure multi-step autoregressive stability).
   - **CNN-LSTM**: Decoupled spatial convolutions (`TimeDistributed`) followed by a 64-unit LSTM.
   - **Pixel-Wise Linear Regression**: Geodetic constant-velocity benchmark using Ordinary Least Squares (OLS).
6. **Recursive Multi-Horizon Forecasting**: Open-loop projections across $t+1$ (12 days), $t+6$ (2.5 months), and $t+17$ (7 months / long-term).
7. **Geodetic Performance Assessment & Spatial Mapping**: Compute $R^2$, MAE (mm), RMSE (mm), Spatial Recall, and export deformation and error residual maps.

In [ ]:
import os
import sys
import gc
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[INFO] GPU Detected & Memory Growth Enabled: {len(gpus)} device(s)")
    except RuntimeError as e:
        print(f"[ERROR] Memory Growth Configuration Failed: {e}")
else:
    print("[WARN] No GPU detected. Execution will proceed on CPU.")

print(f"[INFO] TensorFlow Runtime Version: {tf.__version__}")

## 1. Experimental Configuration & Geodetic Metadata
Define the spatial bounding box for Sayung District ($64 \times 64$ grid resolution), the Lempongsari geodetic reference anchor point, temporal lag ($T=12$), and multi-horizon projection lead times.

In [ ]:
class Config:
    DATA_DIR = "./data"
    OUTPUT_DIR = "./geoai_models"
    TENSOR_INPUT_PATH = os.path.join(DATA_DIR, "sayung_insar_cube_5d.npz")

    TARGET_NORTH = -6.859
    TARGET_SOUTH = -6.987
    TARGET_WEST = 110.478
    TARGET_EAST = 110.566
    GRID_SIZE = 64

    REF_LAT = -6.992
    REF_LON = 110.412
    CALIBRATION_FACTOR = -0.00488247

    LOOKBACK_T = 12
    CHANNELS = 3

    HORIZON_SHORT = 1
    HORIZON_MEDIUM = 6
    HORIZON_LONG = 17

    BATCH_SIZE = 4
    EPOCHS = 100
    LEARNING_RATE = 1e-4

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Artifacts and evaluation checkpoints will be saved to: {Config.OUTPUT_DIR}")

## 2. InSAR 5D Tensor Ingestion & Valid Land Masking
Load the Sentinel-1 SBAS-InSAR 5D tensor $(B, T=12, 64, 64, 3)$ and spatial land mask. Water bodies and decorrelated coastal wetlands are retained as masked pixels to prevent artificial spatial interpolation bias.

In [ ]:
def load_sayung_spatiotemporal_data(file_path):
    """
    Loads preprocessed 5D InSAR tensor and land mask.
    Falls back to a synthetic spatial deformation proxy if local files are not yet downloaded.
    """
    if os.path.exists(file_path):
        print(f"[INFO] Ingesting InSAR 5D tensor from: {file_path}")
        data = np.load(file_path, allow_pickle=True)
        tensor_data = data['tensor']
        targets = data['targets']
        valid_mask = data['valid_mask']
        dates = data.get('dates', None)
    else:
        print(f"[WARN] Local tensor file '{file_path}' not found.")
        print("[INFO] Generating synthetic proxy matching Sayung InSAR velocity & geometry...")
        N, T, H, W, C = 100, Config.LOOKBACK_T, Config.GRID_SIZE, Config.GRID_SIZE, Config.CHANNELS

        valid_mask = np.ones((H, W), dtype=bool)
        valid_mask[:12, :20] = False

        y_coords = np.linspace(0, 1, H)[:, None]
        subsidence_rate = 20.0 + 130.0 * y_coords

        tensor_data = np.zeros((N, T, H, W, C), dtype=np.float32)
        targets = np.zeros((N, H, W, 1), dtype=np.float32)

        for t in range(T):
            t_frac = (t + 1) * (12.0 / 365.25)
            tensor_data[:, t, :, :, 0] = -(subsidence_rate * t_frac) + np.random.normal(0, 2.0, size=(N, H, W))
            tensor_data[:, t, :, :, 1] = 0.65
            tensor_data[:, t, :, :, 2] = 5.0

        t_next = (T + 1) * (12.0 / 365.25)
        targets[:, :, :, 0] = -(subsidence_rate * t_next) + np.random.normal(0, 2.0, size=(N, H, W))

        tensor_data[:, :, ~valid_mask, :] = 0.0
        targets[:, ~valid_mask, :] = 0.0
        dates = [f"2021-01-{i*12:03d}" for i in range(N)]

    return tensor_data, targets, valid_mask, dates

X, y, valid_mask, acquisition_dates = load_sayung_spatiotemporal_data(Config.TENSOR_INPUT_PATH)
print(f"[INFO] Feature Tensor X Shape : {X.shape} (Samples, Timesteps, Height, Width, Channels)")
print(f"[INFO] Target Ground Truth y   : {y.shape} (Samples, Height, Width, 1)")
print(f"[INFO] Valid Terrestrial Pixels: {np.sum(valid_mask)} / {valid_mask.size} ({np.mean(valid_mask)*100:.2f}%)")

## 3. Chronological Dataset Partitioning with Transition Buffers
To prevent artificial temporal autocorrelation leakage, data is partitioned strictly in chronological order matching the thesis methodology:
- **Training**: Jan 2021 – Dec 2024
- *Buffer 1*: Jan 2025 (excluded)
- **Validation**: Feb 2025 – Jul 2025
- *Buffer 2*: Aug 2025 (excluded)
- **Testing**: Sep 2025 – Apr 2026

In [ ]:
N = len(X)
idx_train_end = int(N * 0.70)
idx_val_start = idx_train_end + int(N * 0.03)
idx_val_end = idx_val_start + int(N * 0.12)
idx_test_start = idx_val_end + int(N * 0.03)

X_train, y_train = X[:idx_train_end], y[:idx_train_end]
X_val, y_val = X[idx_val_start:idx_val_end], y[idx_val_start:idx_val_end]
X_test, y_test = X[idx_test_start:], y[idx_test_start:]

print(f"[INFO] Training Samples   : {len(X_train)}")
print(f"[INFO] Validation Samples : {len(X_val)} (after 1-month temporal buffer)")
print(f"[INFO] Testing Samples    : {len(X_test)} (after 1-month temporal buffer)")

## 4. Geodetic Normalization & Masked Objective Formulation
InSAR deformation values represent negative vertical millimeter settlements. Normalization is performed scalar-wise over valid land pixels. A specialized **Masked Loss** ensures gradient updates are calculated strictly on coherent terrestrial ground.

In [ ]:
valid_pixels_disp = X_train[:, :, valid_mask, 0]
MIN_DISP = float(np.min(valid_pixels_disp))
MAX_DISP = float(np.max(valid_pixels_disp))

print(f"[INFO] Physical Subsidence Bounds -> Min: {MIN_DISP:.2f} mm | Max: {MAX_DISP:.2f} mm")

def normalize_tensor(tensor_data):
    norm = tensor_data.copy()
    norm[..., 0] = (norm[..., 0] - MIN_DISP) / (MAX_DISP - MIN_DISP + 1e-8)
    norm[..., 0] = np.nan_to_num(norm[..., 0], nan=0.0)
    return norm

def denormalize_displacement(norm_array):
    return norm_array * (MAX_DISP - MIN_DISP + 1e-8) + MIN_DISP

X_train_norm = normalize_tensor(X_train)
y_train_norm = (y_train - MIN_DISP) / (MAX_DISP - MIN_DISP + 1e-8)

X_val_norm = normalize_tensor(X_val)
y_val_norm = (y_val - MIN_DISP) / (MAX_DISP - MIN_DISP + 1e-8)

X_test_norm = normalize_tensor(X_test)
y_test_norm = (y_test - MIN_DISP) / (MAX_DISP - MIN_DISP + 1e-8)

mask_tensor = tf.constant(valid_mask[..., np.newaxis], dtype=tf.float32)

def masked_mse_loss(y_true, y_pred):
    """Calculates Mean Squared Error strictly over valid terrestrial pixels."""
    diff_sq = tf.square(y_true - y_pred) * mask_tensor
    return tf.reduce_sum(diff_sq) / (tf.reduce_sum(mask_tensor) + 1e-8)

def masked_mae_metric(y_true, y_pred):
    """Calculates Mean Absolute Error strictly over valid terrestrial pixels."""
    diff_abs = tf.abs(y_true - y_pred) * mask_tensor
    return tf.reduce_sum(diff_abs) / (tf.reduce_sum(mask_tensor) + 1e-8)

def evaluate_physical_metrics(y_true_mm, y_pred_mm, mask):
    """Computes R², MAE (mm), and RMSE (mm) exclusively on valid land coordinates."""
    flat_true = y_true_mm[:, mask, 0].flatten()
    flat_pred = y_pred_mm[:, mask, 0].flatten()

    mae = mean_absolute_error(flat_true, flat_pred)
    rmse = np.sqrt(mean_squared_error(flat_true, flat_pred))
    r2 = r2_score(flat_true, flat_pred)
    return {"R2": r2, "MAE_mm": mae, "RMSE_mm": rmse}

## 5. Model 1: ConvLSTM Architecture
The ConvLSTM incorporates $3 \times 3$ convolutions directly inside recurrent memory cells:
- **Layer 1**: 64 filters, `return_sequences=True`
- **Layer 2**: 32 filters, `return_sequences=False`
- **Activation**: Strict `tanh` with `sigmoid` recurrent gates without `BatchNormalization` to guarantee numerical stability across multi-step recursive rollouts.

In [ ]:
def build_convlstm_model(input_shape=(Config.LOOKBACK_T, Config.GRID_SIZE, Config.GRID_SIZE, Config.CHANNELS)):
    inputs = layers.Input(shape=input_shape, name="insar_seq_input")

    x = layers.ConvLSTM2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        return_sequences=True,
        activation="tanh",
        recurrent_activation="sigmoid",
        kernel_regularizer=regularizers.l2(1e-5),
        name="convlstm_64"
    )(inputs)
    x = layers.SpatialDropout3D(0.1, name="spatial_dropout")(x)

    x = layers.ConvLSTM2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        return_sequences=False,
        activation="tanh",
        recurrent_activation="sigmoid",
        kernel_regularizer=regularizers.l2(1e-5),
        name="convlstm_32"
    )(x)

    x = layers.Conv2D(16, (3, 3), padding="same", activation="tanh", name="refinement_conv")(x)
    outputs = layers.Conv2D(1, (1, 1), padding="same", activation="linear", name="displacement_output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="ConvLSTM_Sayung")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=Config.LEARNING_RATE),
        loss=masked_mse_loss,
        metrics=[masked_mae_metric]
    )
    return model

convlstm_model = build_convlstm_model()
convlstm_model.summary()

## 6. Model 2: CNN-LSTM Hybrid Architecture
Decoupled spatial-temporal baseline: uses `TimeDistributed(Conv2D)` layers to extract spatial representations, flattens spatial vectors, and routes them through a 64-unit LSTM.

In [ ]:
def build_cnn_lstm_model(input_shape=(Config.LOOKBACK_T, Config.GRID_SIZE, Config.GRID_SIZE, Config.CHANNELS)):
    inputs = layers.Input(shape=input_shape, name="cnn_lstm_input")

    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), padding="same", activation="relu"))(inputs)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x) # 32 x 32
    x = layers.TimeDistributed(layers.Conv2D(16, (3, 3), padding="same", activation="relu"))(x)
    x = layers.TimeDistributed(layers.Flatten())(x)

    x = layers.LSTM(64, return_sequences=False, name="lstm_64")(x)

    x = layers.Dense(32 * 32 * 8, activation="relu")(x)
    x = layers.Reshape((32, 32, 8))(x)
    x = layers.Conv2DTranspose(16, (3, 3), strides=(2, 2), padding="same", activation="relu")(x) # 64 x 64
    outputs = layers.Conv2D(1, (1, 1), padding="same", activation="linear", name="cnn_lstm_output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_Sayung")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=Config.LEARNING_RATE),
        loss=masked_mse_loss,
        metrics=[masked_mae_metric]
    )
    return model

cnn_lstm_model = build_cnn_lstm_model()
cnn_lstm_model.summary()

## 7. Model 3: Pixel-Wise Linear Regression Baseline
Geodetic baseline fitting an Ordinary Least Squares (OLS) constant-velocity slope ($\text{velocity} = \Delta \text{LOS} / \Delta t$) across each individual valid pixel.

In [ ]:
class PixelWiseLinearRegression:
    """Fits independent constant-velocity OLS linear trends per pixel."""
    def __init__(self, lookback_t=Config.LOOKBACK_T):
        self.lookback_t = lookback_t
        self.time_steps = np.arange(lookback_t).reshape(-1, 1)
        self.time_design = np.hstack([np.ones_like(self.time_steps), self.time_steps]) # [1, t]

    def predict_next_step(self, sequence_window, step_ahead=1):
        """
        sequence_window: (T, H, W) cumulative displacement
        Returns: (H, W) projected cumulative displacement
        """
        T, H, W = sequence_window.shape
        flat_seq = sequence_window.reshape(T, H * W)

        weights, _, _, _ = np.linalg.lstsq(self.time_design, flat_seq, rcond=None)
        intercepts, slopes = weights[0], weights[1]

        target_t = T - 1 + step_ahead
        pred_flat = intercepts + slopes * target_t
        return pred_flat.reshape(H, W)

linear_baseline = PixelWiseLinearRegression()
print("[INFO] Pixel-Wise Linear Regression baseline initialized.")

## 8. Training Pipeline with Model Checkpoints
Train both deep learning models with `EarlyStopping`, `ReduceLROnPlateau`, and automated `.keras` model checkpointing.

In [ ]:
def get_callbacks(model_name):
    return [
        callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(
            filepath=os.path.join(Config.OUTPUT_DIR, f"{model_name}_best.keras"),
            monitor="val_loss",
            save_best_only=True,
            verbose=0
        )
    ]

print("TRAINING CONVLSTM MODEL")
history_convlstm = convlstm_model.fit(
    X_train_norm, y_train_norm,
    validation_data=(X_val_norm, y_val_norm),
    batch_size=Config.BATCH_SIZE,
    epochs=Config.EPOCHS,
    callbacks=get_callbacks("convlstm_sayung"),
    verbose=1
)

print("TRAINING CNN-LSTM HYBRID MODEL")
history_cnn_lstm = cnn_lstm_model.fit(
    X_train_norm, y_train_norm,
    validation_data=(X_val_norm, y_val_norm),
    batch_size=Config.BATCH_SIZE,
    epochs=Config.EPOCHS,
    callbacks=get_callbacks("cnn_lstm_sayung"),
    verbose=1
)

## 9. Recursive Multi-Horizon Forecasting Engine ($t+1, t+6, t+17$)
Simulates autoregressive rollout where previous predictions are fed back into channel 0 of the historical sequence buffer. This evaluates error propagation across short-term, medium-term, and long-term horizons.

In [ ]:
def recursive_rollout_prediction(model, initial_seq_norm, steps=Config.HORIZON_LONG):
    """
    initial_seq_norm shape: (1, T, H, W, C)
    Returns: Array of shape (steps, H, W, 1) in normalized scale
    """
    curr_seq = initial_seq_norm.copy()
    predictions = []

    for s in range(steps):
        next_pred = model.predict(curr_seq, verbose=0) # (1, H, W, 1)
        predictions.append(next_pred[0])

        new_step = np.zeros((1, 1, Config.GRID_SIZE, Config.GRID_SIZE, Config.CHANNELS), dtype=np.float32)
        new_step[0, 0, :, :, 0] = next_pred[0, :, :, 0]
        new_step[0, 0, :, :, 1] = curr_seq[0, -1, :, :, 1]
        new_step[0, 0, :, :, 2] = curr_seq[0, -1, :, :, 2]

        curr_seq = np.concatenate([curr_seq[:, 1:, ...], new_step], axis=1)

    return np.array(predictions)

print("[INFO] Multi-step recursive rollout function defined successfully.")

## 10. Multi-Model Benchmarking Across Lead-Times
Calculate physical metrics ($R^2$, MAE mm, RMSE mm) across the holdout test set for $t+1$ (12 days), $t+6$ (2.5 months), and $t+17$ (7 months).

In [ ]:
horizons_to_test = [Config.HORIZON_SHORT, Config.HORIZON_MEDIUM, Config.HORIZON_LONG]
benchmark_summary = []

for h in horizons_to_test:
    valid_test_count = len(X_test) - h + 1
    if valid_test_count <= 0:
        continue

    conv_preds_mm = []
    cnn_preds_mm = []
    linear_preds_mm = []
    actuals_mm = []

    for i in range(valid_test_count):
        init_seq_norm = X_test_norm[i:i+1]
        rollout_conv = recursive_rollout_prediction(convlstm_model, init_seq_norm, h)
        rollout_cnn = recursive_rollout_prediction(cnn_lstm_model, init_seq_norm, h)

        conv_preds_mm.append(denormalize_displacement(rollout_conv[h-1]))
        cnn_preds_mm.append(denormalize_displacement(rollout_cnn[h-1]))

        raw_disp_seq = X_test[i, :, :, :, 0]
        pred_lin = linear_baseline.predict_next_step(raw_disp_seq, step_ahead=h)
        linear_preds_mm.append(pred_lin[..., np.newaxis])

        actuals_mm.append(y_test[i + h - 1])

    actuals_arr = np.array(actuals_mm)
    conv_arr = np.array(conv_preds_mm)
    cnn_arr = np.array(cnn_preds_mm)
    lin_arr = np.array(linear_preds_mm)

    m_conv = evaluate_physical_metrics(actuals_arr, conv_arr, valid_mask)
    m_cnn = evaluate_physical_metrics(actuals_arr, cnn_arr, valid_mask)
    m_lin = evaluate_physical_metrics(actuals_arr, lin_arr, valid_mask)

    lead_name = f"t+{h} ({'12 days' if h==1 else '2.5 mos' if h==6 else '7 mos'})"
    benchmark_summary.append({
        "Horizon": lead_name,
        "ConvLSTM_R2": round(m_conv["R2"], 4),
        "ConvLSTM_MAE (mm)": round(m_conv["MAE_mm"], 2),
        "CNN-LSTM_R2": round(m_cnn["R2"], 4),
        "CNN-LSTM_MAE (mm)": round(m_cnn["MAE_mm"], 2),
        "LinearReg_R2": round(m_lin["R2"], 4),
        "LinearReg_MAE (mm)": round(m_lin["MAE_mm"], 2),
    })

df_benchmark = pd.DataFrame(benchmark_summary)
print("BENCHMARKING PERFORMANCE TABLE")
print(df_benchmark.to_string(index=False))

df_benchmark.to_csv(os.path.join(Config.OUTPUT_DIR, "table_1_benchmarking_results.csv"), index=False)

## 11. Spatial Deformation & Residual Error Mapping
Map actual cumulative land subsidence against ConvLSTM forecasts across Sayung District, alongside spatial residual maps ($|\text{Actual} - \text{Predicted}|$).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row_idx, h in enumerate([Config.HORIZON_SHORT, Config.HORIZON_MEDIUM]):
    actual_field = np.where(valid_mask, y_test[h-1, :, :, 0], np.nan)

    pred_field = np.where(valid_mask, denormalize_displacement(recursive_rollout_prediction(convlstm_model, X_test_norm[0:1], h)[h-1, :, :, 0]), np.nan)
    residual_field = np.where(valid_mask, np.abs(actual_field - pred_field), np.nan)

    vmin, vmax = np.nanpercentile(actual_field, 2), np.nanpercentile(actual_field, 98)

    im0 = axes[row_idx, 0].imshow(actual_field, cmap="turbo_r", vmin=vmin, vmax=vmax)
    axes[row_idx, 0].set_title(f"Observed Subsidence (t+{h})", fontweight="bold")
    fig.colorbar(im0, ax=axes[row_idx, 0], label="Deformation (mm)")

    im1 = axes[row_idx, 1].imshow(pred_field, cmap="turbo_r", vmin=vmin, vmax=vmax)
    axes[row_idx, 1].set_title(f"ConvLSTM Projection (t+{h})", fontweight="bold")
    fig.colorbar(im1, ax=axes[row_idx, 1], label="Deformation (mm)")

    im2 = axes[row_idx, 2].imshow(residual_field, cmap="magma", vmin=0, vmax=25)
    axes[row_idx, 2].set_title(f"Absolute Residual |t+{h}|", fontweight="bold")
    fig.colorbar(im2, ax=axes[row_idx, 2], label="Error (mm)")

    for ax in axes[row_idx]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Spatiotemporal Subsidence Projections & Spatial Residuals (Sayung, Demak)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, "spatial_residuals_sayung.png"), dpi=300)
plt.show()